In [1]:
# ============================================================
# SenseForge — Notebook 20
# End-to-End Multilingual NLP Integration
# Cell 1: Model loading and inference engine
# ============================================================

from pathlib import Path
import sys
import re
import json
import gc
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# ------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from senseforge.core import INVENTORY, predict as rule_predict

ART = ROOT / "artifacts"
LORA_DIR = ART / "wsd_v2_lora"
LID_DIR = (
    ART / "senseforge_final" / "comilingua_lid"
    / "training_v1" / "normalized_3class"
)
MT_DIR = ART / "translation_v1" / "finetuned" / "best_model"
HYBRID_CONFIG = ART / "hybrid_v2" / "hybrid_config.json"

OUT = ART / "senseforge_end_to_end"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

WSD_MAX_LENGTH = 64
MT_MAX_LENGTH = 96

SENSES = [
    "bank.1", "bank.2",
    "interest.1", "interest.2",
    "plant.1", "plant.2",
    "point.1", "point.2",
    "right.1", "right.2",
    "table.1", "table.2"
]

NEURAL_WORDS = {
    s.rsplit(".", 1)[0] for s in SENSES
}

config = json.loads(
    HYBRID_CONFIG.read_text(encoding="utf-8")
)
THRESHOLD = float(config["threshold"])

print("=== SENSEFORGE MODEL LOADING ===")
print("Project:", ROOT)
print("Device:", DEVICE)

# ------------------------------------------------------------
# Load Comi-Lingua LID
# ------------------------------------------------------------

lid_path = LID_DIR / "comilingua_lid_3class.joblib"
assert lid_path.exists(), f"Missing LID model: {lid_path}"

lid_model = joblib.load(lid_path)

print("LID model loaded.")

# ------------------------------------------------------------
# Load LoRA WSD
# ------------------------------------------------------------

class LoRAWSDClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes)
        )

    def forward(
        self, input_ids, attention_mask,
        target_mask, class_mask=None
    ):
        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        weights = target_mask.unsqueeze(-1)
        pooled = (
            (hidden * weights).sum(dim=1)
            / weights.sum(dim=1).clamp(min=1)
        )

        logits = self.classifier(self.dropout(pooled))

        if class_mask is not None:
            logits = logits.masked_fill(
                ~class_mask.bool(), -1e4
            )

        return logits


wsd_tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base", use_fast=True
)

base_encoder = AutoModel.from_pretrained(
    "xlm-roberta-base",
    attn_implementation="eager"
)

encoder = PeftModel.from_pretrained(
    base_encoder,
    LORA_DIR / "best_lora_adapter",
    is_trainable=False
)

wsd_model = LoRAWSDClassifier(encoder, len(SENSES))

head = torch.load(
    LORA_DIR / "best_classifier_head.pt",
    map_location="cpu",
    weights_only=True
)

wsd_model.classifier.load_state_dict(head)
wsd_model.eval().to("cpu")

print("XLM-R LoRA WSD loaded.")

# ------------------------------------------------------------
# Load translation model
# ------------------------------------------------------------
# The translation model is loaded on demand to keep
# memory usage manageable on an 8 GB Mac.

mt_tokenizer = None
mt_model = None

def load_translation_model():
    global mt_tokenizer, mt_model

    if mt_model is None:
        assert MT_DIR.exists(), f"Missing translation model: {MT_DIR}"

        mt_tokenizer = AutoTokenizer.from_pretrained(
            MT_DIR
        )

        mt_model = AutoModelForSeq2SeqLM.from_pretrained(
            MT_DIR
        ).to(DEVICE)

        mt_model.eval()
        print("Fine-tuned MarianMT loaded.")

    return mt_model


# ------------------------------------------------------------
# Token-level language identification
# ------------------------------------------------------------
# The classifier uses individual token strings.
# Whitespace tokenization is an inference convenience,
# not a reconstruction of Comi-Lingua's gold tokenizer.

def identify_languages(text):
    matches = list(re.finditer(r"\S+", text))
    tokens = [m.group() for m in matches]

    if not tokens:
        return []

    predictions = lid_model.predict(tokens)

    return [
        {
            "token": token,
            "language": str(label),
            "start": match.start(),
            "end": match.end()
        }
        for token, label, match in zip(
            tokens, predictions, matches
        )
    ]


# ------------------------------------------------------------
# Locate the ambiguous target
# ------------------------------------------------------------

def locate_target(text, word, start=None, end=None):
    if start is not None or end is not None:
        if start is None or end is None:
            raise ValueError("Provide both offsets")

        start, end = int(start), int(end)

        if not 0 <= start < end <= len(text):
            raise ValueError("Invalid target span")

        return start, end

    matches = list(re.finditer(
        rf"(?<!\w){re.escape(word)}(?!\w)",
        text, flags=re.IGNORECASE
    ))

    if len(matches) != 1:
        raise ValueError(
            f"Found {len(matches)} occurrences of '{word}'. "
            "Provide target offsets."
        )

    return matches[0].span()


# ------------------------------------------------------------
# Target-centered transformer encoding
# ------------------------------------------------------------
# This follows the target-aware 64-token setup used
# in our existing LoRA evaluation notebooks.

def encode_wsd(text, word, start, end):
    encoded = wsd_tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_offsets_mapping=True
    )

    ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    positions = [
        i for i, (a, b) in enumerate(offsets)
        if b > a and a < end and b > start
    ]

    if not positions:
        raise ValueError("Target-token alignment failed")

    capacity = WSD_MAX_LENGTH - 2
    first, last = positions[0], positions[-1]

    if last - first + 1 > capacity:
        raise ValueError("Target exceeds context window")

    left = max(1, first - capacity // 2)
    left = min(left, max(1, last - capacity + 1))

    right = min(len(ids) - 1, left + capacity)

    if right - left < capacity:
        left = max(1, right - capacity)

    selected = list(range(left, right))
    target_set = set(positions)

    input_ids = (
        [ids[0]]
        + [ids[i] for i in selected]
        + [ids[-1]]
    )

    target_mask = (
        [0.0]
        + [float(i in target_set) for i in selected]
        + [0.0]
    )

    attention = [1] * len(input_ids)
    padding = WSD_MAX_LENGTH - len(input_ids)

    assert padding >= 0
    assert sum(target_mask) > 0

    input_ids += [wsd_tokenizer.pad_token_id] * padding
    attention += [0] * padding
    target_mask += [0.0] * padding

    allowed = [
        s.startswith(word + ".") for s in SENSES
    ]

    assert sum(allowed) == 2

    return {
        "input_ids": torch.tensor(
            [input_ids], dtype=torch.long
        ),
        "attention_mask": torch.tensor(
            [attention], dtype=torch.long
        ),
        "target_mask": torch.tensor(
            [target_mask], dtype=torch.float32
        ),
        "class_mask": torch.tensor(
            [allowed], dtype=torch.bool
        )
    }


@torch.inference_mode()
def neural_wsd(text, word, start, end):
    batch = encode_wsd(text, word, start, end)
    logits = wsd_model(**batch)

    probs = torch.softmax(logits, dim=-1)[0]
    confidence, index = probs.max(dim=0)

    return (
        SENSES[index.item()],
        float(confidence.item())
    )


# ------------------------------------------------------------
# Rule-based inference
# ------------------------------------------------------------

def rule_wsd(text, word, start, end):
    # Normalize the annotated target to its lemma so
    # inflected forms remain compatible with the old rules.

    normalized_text = text[:start] + word + text[end:]

    return rule_predict(
        normalized_text,
        word,
        method="rule"
    )


# ------------------------------------------------------------
# English-to-Hindi translation
# ------------------------------------------------------------

@torch.inference_mode()
def translate_en_hi(text):
    translation_model = load_translation_model()

    encoded = mt_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MT_MAX_LENGTH
    ).to(DEVICE)

    generated = translation_model.generate(
        **encoded,
        max_new_tokens=MT_MAX_LENGTH,
        num_beams=2
    )

    return mt_tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )


# ------------------------------------------------------------
# Unified SenseForge prediction
# ------------------------------------------------------------

def senseforge_predict(
    text,
    target_word,
    start=None,
    end=None,
    include_translation=True
):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input sentence cannot be empty")

    word = target_word.lower().strip()

    start, end = locate_target(
        text, word, start, end
    )

    token_languages = identify_languages(text)

    warnings_list = []

    # This is only a script-based input check.
    # It is not a validated sentence-level LID decision.
    has_devanagari = bool(
        re.search(r"[\u0900-\u097F]", text)
    )

    if has_devanagari:
        warnings_list.append(
            "Code-mixed/Hindi input: English WSD accuracy "
            "has not been validated on this input type."
        )

    sense_id = None
    hindi = None
    definition = None
    confidence = None
    method = "unsupported"
    alternatives = []

    if word in INVENTORY:
        base = rule_wsd(text, word, start, end)

        sense_id = base["sense_id"]
        method = "rule_fallback"
        alternatives = base.get("alternatives", [])

        if word in NEURAL_WORDS:
            neural_label, confidence = neural_wsd(
                text, word, start, end
            )

            if confidence >= THRESHOLD:
                sense_id = neural_label
                method = "hybrid_lora"

        selected = next(
            s for s in INVENTORY[word]["senses"]
            if s["id"] == sense_id
        )

        hindi = selected["hindi"]
        definition = selected["definition"]

    else:
        warnings_list.append(
            "Target word is not in the WSD inventory."
        )

    translation = None

    if include_translation:
        if has_devanagari:
            warnings_list.append(
                "English-to-Hindi translation skipped "
                "for input containing Devanagari."
            )
        else:
            translation = translate_en_hi(text)

    return {
        "input_text": text,
        "target_word": word,
        "target_surface": text[start:end],
        "target_span": [start, end],
        "token_languages": token_languages,
        "sense_id": sense_id,
        "hindi_meaning": hindi,
        "definition": definition,
        "prediction_method": method,
        "neural_confidence": confidence,
        "confidence_calibrated": False,
        "alternatives": alternatives,
        "translation_hi": translation,
        "translation_model": (
            "finetuned_marianmt" if translation is not None
            else None
        ),
        "warnings": warnings_list
    }


print("\n=== INTEGRATION READY ===")
print("Neural WSD words:", sorted(NEURAL_WORDS))
print("Inventory words:", len(INVENTORY))
print("Hybrid threshold:", THRESHOLD)
print("Translation: lazy loading enabled")

=== SENSEFORGE MODEL LOADING ===
Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Device: mps
LID model loaded.
XLM-R LoRA WSD loaded.

=== INTEGRATION READY ===
Neural WSD words: ['bank', 'interest', 'plant', 'point', 'right', 'table']
Inventory words: 35
Hybrid threshold: 0.75
Translation: lazy loading enabled


In [3]:
# ============================================================
# SenseForge — Notebook 20
# Cell 2: End-to-end tests, demonstration and export
# ============================================================

import json
import time
from collections import Counter

import pandas as pd

print("=" * 65)
print("SENSEFORGE — END-TO-END INTEGRATION TEST")
print("=" * 65)

# ------------------------------------------------------------
# Demonstration examples
# ------------------------------------------------------------
# Each example checks a different part of the pipeline:
# financial/river senses, industrial/biological meanings,
# directional ambiguity, rule fallback and mixed language.

examples = [
    {
        "text": "The bank approved the loan.",
        "word": "bank",
        "expected": "bank.1"
    },
    {
        "text": "The fisherman sat on the bank.",
        "word": "bank",
        "expected": "bank.2"
    },
    {
        "text": "The plant needs sunlight.",
        "word": "plant",
        "expected": "plant.1"
    },
    {
        "text": "The factory plant closed yesterday.",
        "word": "plant",
        "expected": "plant.2"
    },
    {
        "text": "Turn right after the bridge.",
        "word": "right",
        "expected": "right.2"
    },
    {
        "text": "The answer is right.",
        "word": "right",
        "expected": "right.1"
    },
    {
        "text": "The table contains several rows.",
        "word": "table",
        "expected": "table.2"
    },
    {
        "text": "She placed the book on the table.",
        "word": "table",
        "expected": "table.1"
    },
    {
        "text": "A bat flew out of the cave.",
        "word": "bat",
        "expected": "bat.2"
    },
    {
        "text": "He hit the ball with a bat.",
        "word": "bat",
        "expected": "bat.1"
    }
]

results = []
start_time = time.perf_counter()

# First evaluate WSD without translation.
# This isolates sense-prediction errors from translation errors.

for example in examples:
    result = senseforge_predict(
        example["text"],
        example["word"],
        include_translation=False
    )

    result["expected_sense"] = example["expected"]
    result["correct"] = (
        result["sense_id"] == example["expected"]
    )

    results.append(result)

    print(
        f"{example['word']:10} | "
        f"{str(result['sense_id']):12} | "
        f"{str(result['hindi_meaning']):12} | "
        f"{result['prediction_method']:15} | "
        f"{'PASS' if result['correct'] else 'FAIL'}"
    )

correct = sum(r["correct"] for r in results)

print("\nWSD demonstrations correct:", correct, "/", len(results))

# ------------------------------------------------------------
# Metadata consistency checks
# ------------------------------------------------------------

for result in results:
    word = result["target_word"]
    sense_id = result["sense_id"]

    assert sense_id.startswith(word + ".")

    selected = next(
        s for s in INVENTORY[word]["senses"]
        if s["id"] == sense_id
    )

    assert result["hindi_meaning"] == selected["hindi"]
    assert result["definition"] == selected["definition"]

print("Hindi meaning consistency: PASSED")

# ------------------------------------------------------------
# Multiple-occurrence target test
# ------------------------------------------------------------

text = "The bank called the bank manager."
second_start = text.rfind("bank")

multiple_result = senseforge_predict(
    text,
    "bank",
    start=second_start,
    end=second_start + 4,
    include_translation=False
)

assert multiple_result["target_span"] == [
    second_start, second_start + 4
]

print("Explicit target offsets: PASSED")

# ------------------------------------------------------------
# Verify ambiguous occurrences are rejected
# ------------------------------------------------------------

try:
    senseforge_predict(
        "The bank called the bank manager.",
        "bank",
        include_translation=False
    )
except ValueError:
    print("Ambiguous occurrence handling: PASSED")
else:
    raise AssertionError(
        "Multiple target occurrences were accepted "
        "without explicit offsets"
    )

# ------------------------------------------------------------
# Code-mixed input demonstration
# ------------------------------------------------------------
# This is a functionality test, not a validated Hinglish
# WSD benchmark.

mixed_result = senseforge_predict(
    "Mujhe bank जाना है।",
    "bank",
    include_translation=False
)

print("\n=== CODE-MIXED INPUT ===")
print("Sense:", mixed_result["sense_id"])
print("Hindi meaning:", mixed_result["hindi_meaning"])
print("Warnings:", mixed_result["warnings"])

# ------------------------------------------------------------
# Translation integration
# ------------------------------------------------------------
# We load the saved MarianMT checkpoint only now.
# The generated translation is shown as a demonstration,
# not automatically marked correct.

translation_result = senseforge_predict(
    "The bank approved the loan.",
    "bank",
    include_translation=True
)

print("\n=== TRANSLATION INTEGRATION ===")
print("English:", translation_result["input_text"])
print("Hindi:", translation_result["translation_hi"])
print("Sense:", translation_result["sense_id"])
print("Meaning:", translation_result["hindi_meaning"])

assert isinstance(
    translation_result["translation_hi"], str
)
assert translation_result["translation_hi"].strip()

# ------------------------------------------------------------
# Save reports
# ------------------------------------------------------------

def write_json(path, data):
    path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
            default=str
        ),
        encoding="utf-8"
    )

write_json(
    OUT / "integration_demo_results.json",
    results
)

write_json(
    OUT / "code_mixed_demo.json",
    mixed_result
)

write_json(
    OUT / "translation_demo.json",
    translation_result
)

summary = {
    "project": "SenseForge",
    "notebook": "20_SenseForge_End_to_End_Integration",
    "wsd_demonstrations": len(results),
    "wsd_correct": correct,
    "wsd_demo_accuracy": correct / len(results),
    "metadata_consistency": "passed",
    "multiple_occurrence_test": "passed",
    "code_mixed_demo": "executed",
    "translation_demo": "executed",
    "lid_model": "comilingua_3class_tfidf_logreg",
    "wsd_model": "xlmr_lora_with_rule_fallback",
    "translation_model": "finetuned_marianmt",
    "runtime_seconds": round(
        time.perf_counter() - start_time, 2
    ),
    "limitations": [
        "WSD is validated primarily on English contexts.",
        "Code-mixed WSD accuracy is not established.",
        "LID uses token-only features and provisional label mapping.",
        "Translation quality remains limited.",
        "Neural WSD supports six ambiguous English lemmas.",
        "Confidence scores are not calibrated.",
        "Integration demos are not independent benchmarks."
    ]
}

write_json(
    OUT / "integration_summary.json",
    summary
)

print("\n" + "=" * 65)
print("SENSEFORGE INTEGRATION RUN COMPLETE")
print("=" * 65)

print("WSD demonstrations:", correct, "/", len(results))
print("LID checkpoint: loaded")
print("WSD checkpoint: loaded")
print("Translation checkpoint: loaded")
print("Outputs:", OUT)

print("\nInference API:")
print(
    'senseforge_predict('
    '"The fisherman sat on the bank.", "bank")'
)

SENSEFORGE — END-TO-END INTEGRATION TEST
bank       | bank.1       | बैंक         | hybrid_lora     | PASS
bank       | bank.2       | किनारा       | hybrid_lora     | PASS
plant      | plant.1      | पौधा         | rule_fallback   | PASS
plant      | plant.2      | कारखाना      | hybrid_lora     | PASS
right      | right.2      | दायाँ        | hybrid_lora     | PASS
right      | right.1      | सही          | hybrid_lora     | PASS
table      | table.2      | तालिका       | hybrid_lora     | PASS
table      | table.1      | मेज़         | hybrid_lora     | PASS
bat        | bat.2        | चमगादड़      | rule_fallback   | PASS
bat        | bat.1        | बल्ला        | rule_fallback   | PASS

WSD demonstrations correct: 10 / 10
Hindi meaning consistency: PASSED
Explicit target offsets: PASSED
Ambiguous occurrence handling: PASSED

=== CODE-MIXED INPUT ===
Sense: bank.1
Hindi meaning: बैंक
Warnings: ['Code-mixed/Hindi input: English WSD accuracy has not been validated on this input type